In [1]:
import re, pickle, json, zipfile, math, time
from collections import defaultdict
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

device = "cuda"

In [2]:
# ==========================================================
# Tokenizer
# ==========================================================
def merge_pair_in_word(word_ids, pair_to_merge, new_id):
    a, b = pair_to_merge
    new_word, i = [], 0
    while i < len(word_ids):
        if i < len(word_ids) - 1 and word_ids[i] == a and word_ids[i + 1] == b:
            new_word.append(new_id); i += 2
        else:
            new_word.append(word_ids[i]); i += 1
    return new_word

class BPETokenizer:
    def __init__(self, vocab, merges):
        self.vocab, self.merges = vocab, merges
        self.pat = re.compile(r" ?[a-zA-Z]+| ?[0-9]+| ?[^\w\s]+|\s+")

    def _encode_chunk(self, chunk_str):
        word_ids = [b + 4 for b in chunk_str.encode("utf-8")]
        while len(word_ids) >= 2:
            pairs = [(word_ids[i], word_ids[i + 1]) for i in range(len(word_ids) - 1)]
            best_pair = min(pairs, key=lambda p: self.merges.get(p, float("inf")))
            if best_pair not in self.merges:
                break
            word_ids = merge_pair_in_word(word_ids, best_pair, self.merges[best_pair])
        return word_ids

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        for chunk in self.pat.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, token_ids):
        return b"".join(self.vocab[t] for t in token_ids).decode("utf-8", errors="replace")

def load_bpe_tokenizer():
    with open("/kaggle/input/datasets/kabeersinghal/bpeer-ra-100000/bpe_checkpoint.pkl", "rb") as f:
        ckpt = pickle.load(f)
    return BPETokenizer(ckpt["vocab"], ckpt["merges"])

tokenizer = load_bpe_tokenizer()

In [3]:
SEP = tokenizer.encode("####")[0]
START = SEP                                   # decoder start token, as in training
END = tokenizer.encode("###")[0]
print("START/SEP", SEP, "END", END)           # should be different ids

START/SEP 6819 END 22132


In [4]:
# ==========================================================
# Model (exactly as trained)
# ==========================================================
@dataclass
class EncDecConfig:
    vocab_size: int
    d_model: int = 960
    n_layers: int = 4
    n_heads: int = 8
    d_ff: int = 1664
    max_seq_len: int = 1024

def sinusoidal_positional_encoding(max_len, d_model):
    PE = torch.zeros(max_len, d_model)
    for i in range(max_len):
        for j in range(d_model):
            if j % 2 == 0:
                PE[i, j] = math.sin(i / (10000 ** (j / d_model)))
            else:
                PE[i, j] = math.cos(i / (10000 ** ((j - 1) / d_model)))
    return PE

class BidirectionalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.d_model = n_heads, d_model
        self.d_head = int(d_model / n_heads)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, key_mask=None):
        B, S, D = x.shape
        q = self.q_proj(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        att = q @ k.transpose(2, 3) / math.sqrt(self.d_head)
        if key_mask is not None:
            att = att.masked_fill(~key_mask[:, None, None, :], float("-inf"))
        att = torch.softmax(att, dim=3)
        return self.out_proj((att @ v).transpose(1, 2).reshape(B, S, D))

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len):
        super().__init__()
        self.n_heads, self.d_model = n_heads, d_model
        self.d_head = int(d_model / n_heads)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.register_buffer("causal_mask",
                             torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
                             persistent=False)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        att = q @ k.transpose(2, 3) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        att = torch.softmax(att, dim=3)
        return self.out_proj((att @ v).transpose(1, 2).reshape(B, T, D))

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.d_model = n_heads, d_model
        self.d_head = int(d_model / n_heads)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_dec, enc_out, enc_mask=None):
        B, T, D = x_dec.shape
        S = enc_out.shape[1]
        q = self.q_proj(x_dec).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(enc_out).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        att = q @ k.transpose(2, 3) / math.sqrt(self.d_head)
        if enc_mask is not None:
            att = att.masked_fill(~enc_mask[:, None, None, :], float("-inf"))
        att = torch.softmax(att, dim=3)
        return self.out_proj((att @ v).transpose(1, 2).reshape(B, T, D))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = BidirectionalSelfAttention(cfg.d_model, cfg.n_heads)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
    def forward(self, x, key_mask=None):
        x = x + self.attn(self.ln1(x), key_mask)
        return x + self.ffn(self.ln2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn1 = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.attn2 = CrossAttention(cfg.d_model, cfg.n_heads)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.ln3 = nn.LayerNorm(cfg.d_model)
    def forward(self, x, enc_out, enc_mask=None):
        x = x + self.attn1(self.ln1(x))
        x = x + self.attn2(self.ln2(x), enc_out, enc_mask)
        return x + self.ffn(self.ln3(x))

class EncoderDecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model),
                             persistent=False)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)
        self.decoder_blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def encode(self, enc_idx, enc_mask=None):
        x = self.tok_emb(enc_idx) + self.pos_emb[:enc_idx.shape[1]]
        for block in self.encoder_blocks:
            x = block(x, enc_mask)
        return self.encoder_ln(x)

    def decode(self, dec_idx, encoder_out, enc_mask=None):
        x = self.tok_emb(dec_idx) + self.pos_emb[:dec_idx.shape[1]]
        for block in self.decoder_blocks:
            x = block(x, encoder_out, enc_mask)
        return self.head(self.decoder_ln(x))

    def forward(self, enc_idx, dec_idx, enc_mask=None):
        return self.decode(dec_idx, self.encode(enc_idx, enc_mask), enc_mask)

In [5]:
# ---------- load weights ----------
CKPT = "/kaggle/input/datasets/whitequotes/p6-enc-ouutput/p6_encdec_ft5.pt"          # <-- set to the epoch you want
cfg = EncDecConfig(vocab_size=50260)
model = EncoderDecoderLM(cfg).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()

test = pickle.load(open("/kaggle/input/datasets/whitequotes/p6-test-processed-dataer/test_data_processed.pkl", "rb"))
# test = test[:50]     # quick toy run

def get_src(e):                                     # article tokens, no SEP appended
    return e["tokens"] if "tokens" in e else e["src"]

In [6]:
# ==========================================================
# Cached generation
#   - encoder runs once per batch (padded, with key mask)
#   - cross-attention K/V computed once per batch
#   - decoder self-attention uses a KV cache, one token per step
# ==========================================================
USE_FP16 = False
USE_REP_RULES = True      # repetition penalty + 3-gram ban from your earlier runs; False = plain greedy
BS = 16

def dec_step(tok, pos, caches, cross, emask):
    """tok (B,1) at decoder position pos. Returns logits (B, V)."""
    B = tok.shape[0]
    h = model.tok_emb(tok) + model.pos_emb[pos:pos + 1]
    for li, blk in enumerate(model.decoder_blocks):
        a = blk.attn1
        z = blk.ln1(h)
        q = a.q_proj(z).view(B, 1, a.n_heads, a.d_head).transpose(1, 2)
        k = a.k_proj(z).view(B, 1, a.n_heads, a.d_head).transpose(1, 2)
        v = a.v_proj(z).view(B, 1, a.n_heads, a.d_head).transpose(1, 2)
        if caches[li] is not None:
            k = torch.cat([caches[li][0], k], dim=2)
            v = torch.cat([caches[li][1], v], dim=2)
        caches[li] = (k, v)
        o = F.scaled_dot_product_attention(q, k, v)             # single query sees all past keys
        h = h + a.out_proj(o.transpose(1, 2).reshape(B, 1, -1))

        c = blk.attn2
        z = blk.ln2(h)
        q = c.q_proj(z).view(B, 1, c.n_heads, c.d_head).transpose(1, 2)
        ck, cv = cross[li]
        o = F.scaled_dot_product_attention(q, ck, cv, attn_mask=emask)   # hide encoder padding
        h = h + c.out_proj(o.transpose(1, 2).reshape(B, 1, -1))

        h = h + blk.ffn(blk.ln3(h))
    return model.head(model.decoder_ln(h[:, -1]))

@torch.no_grad()
def generate(src_lists, max_new):
    B = len(src_lists)
    S = max(len(s) for s in src_lists)
    src = torch.zeros(B, S, dtype=torch.long, device=device)
    mask = torch.zeros(B, S, dtype=torch.bool, device=device)
    for i, s in enumerate(src_lists):
        src[i, :len(s)] = torch.tensor(s, device=device)
        mask[i, :len(s)] = True
    max_new = min(max_new, cfg.max_seq_len - 1)                 # decoder positions: START + max_new <= 1024

    out = torch.zeros(B, max_new, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    rows = torch.arange(B, device=device)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
        enc = model.encode(src, mask)
        cross = []
        for blk in model.decoder_blocks:
            c = blk.attn2
            ck = c.k_proj(enc).view(B, S, c.n_heads, c.d_head).transpose(1, 2)
            cv = c.v_proj(enc).view(B, S, c.n_heads, c.d_head).transpose(1, 2)
            cross.append((ck, cv))
        emask = mask[:, None, None, :]
        caches = [None] * len(model.decoder_blocks)

        tok = torch.full((B, 1), START, dtype=torch.long, device=device)
        logits = dec_step(tok, 0, caches, cross, emask)         # START sits at decoder position 0
        for n in range(max_new):
            logits = logits.float()
            if USE_REP_RULES:
                if n >= 1:
                    recent = out[:, max(0, n - 50):n]
                    vals = logits.gather(1, recent)
                    logits.scatter_(1, recent, torch.where(vals > 0, vals / 1.2, vals * 1.2))
                    logits[rows, out[:, n - 1]] = float("-inf")
                if n >= 3:
                    match = (out[:, :n - 2] == out[:, n - 2:n - 1]) & (out[:, 1:n - 1] == out[:, n - 1:n])
                    ban = torch.zeros_like(logits).scatter_reduce_(
                        1, out[:, 2:n], match.float(), reduce="amax", include_self=True)
                    logits.masked_fill_(ban > 0, float("-inf"))
            nxt = logits.argmax(-1)
            out[:, n] = nxt
            finished |= (nxt == END)
            if finished.all() or n == max_new - 1:
                break
            logits = dec_step(nxt[:, None], n + 1, caches, cross, emask)   # token n sits at position n+1
    return [r[:r.index(END)] if END in r else r for r in out[:, :n + 1].tolist()]

# ---------- sanity check (set USE_FP16 = False first): cached path == model.decode ----------
@torch.no_grad()
def sanity_check(steps=8):
    s = get_src(test[0])[:cfg.max_seq_len]
    src = torch.tensor([s], device=device)
    mask = torch.ones_like(src, dtype=torch.bool)
    enc = model.encode(src, mask)
    cross = []
    for blk in model.decoder_blocks:
        c = blk.attn2
        cross.append((c.k_proj(enc).view(1, -1, c.n_heads, c.d_head).transpose(1, 2),
                      c.v_proj(enc).view(1, -1, c.n_heads, c.d_head).transpose(1, 2)))
    emask = mask[:, None, None, :]
    caches = [None] * len(model.decoder_blocks)
    dec = torch.tensor([[START]], device=device)
    got = dec_step(dec, 0, caches, cross, emask).float()
    for i in range(steps):
        ref = model.decode(dec, enc, mask)[:, -1].float()
        print(f"step {i}  max abs diff:", (ref - got).abs().max().item())
        nxt = ref.argmax(-1, keepdim=True)
        dec = torch.cat([dec, nxt], 1)
        got = dec_step(nxt, dec.shape[1] - 1, caches, cross, emask).float()
sanity_check()     # all diffs should be ~1e-4 or smaller

step 0  max abs diff: 2.86102294921875e-06
step 1  max abs diff: 4.76837158203125e-06
step 2  max abs diff: 4.76837158203125e-06
step 3  max abs diff: 3.814697265625e-06
step 4  max abs diff: 8.58306884765625e-06
step 5  max abs diff: 1.049041748046875e-05
step 6  max abs diff: 1.049041748046875e-05
step 7  max abs diff: 8.106231689453125e-06


In [7]:
# ==========================================================
# Summarization
# ==========================================================
CHUNKED = False            # False: one pass per document. True: chunked protocol like your decoder-only runs
TRUNC = "head"             # for single-pass: keep the "head" or the "tail" of articles longer than MAX_SRC
MAX_SRC, MAX_NEW = 1024, 750
WINDOW, MAX_CHUNKS, TARGET = 1024, 10, 750      # chunked settings

def make_chunks(tokens):
    if len(tokens) <= WINDOW:
        return [tokens]
    chunks = [tokens[s:s + WINDOW] for s in range(0, len(tokens) - WINDOW, WINDOW)]
    chunks.append(tokens[-WINDOW:])
    if len(chunks) > MAX_CHUNKS:
        idx = [round(i * (len(chunks) - 1) / (MAX_CHUNKS - 1)) for i in range(MAX_CHUNKS)]
        chunks = [chunks[i] for i in idx]
    return chunks

def summarize_all(docs):
    items = []                                      # (doc, chunk, src tokens, budget)
    for di, toks in enumerate(docs):
        if CHUNKED:
            chunks = make_chunks(toks)
            budget = max(60, min(300, TARGET // len(chunks)))
        else:
            chunks = [toks[:MAX_SRC] if TRUNC == "head" else toks[-MAX_SRC:]]
            budget = MAX_NEW
        for cj, c in enumerate(chunks):
            items.append((di, cj, c, budget))

    order = sorted(range(len(items)), key=lambda k: len(items[k][2]))   # similar lengths together
    pieces = {}
    with tqdm(total=len(items), desc="Generating", unit="item") as pbar:
        for s in range(0, len(order), BS):
            b = order[s:s + BS]
            outs = generate([items[k][2] for k in b], max_new=max(items[k][3] for k in b))
            for k, o in zip(b, outs):
                di, cj, _, budget = items[k]
                text = tokenizer.decode(o[:budget]).strip()
                if CHUNKED and len(o) >= budget:    # cut by budget: trim to last full sentence
                    cut = text.rfind(".")
                    if cut > len(text) * 0.5:
                        text = text[:cut + 1]
                pieces[(di, cj)] = text
            pbar.update(len(b))

    by_doc = defaultdict(list)
    for (d, j) in pieces:
        by_doc[d].append(j)
    results = []
    for di in range(len(docs)):
        seen, kept = set(), []
        for cj in sorted(by_doc[di]):
            for sent in re.split(r"(?<=\.)\s+", pieces[(di, cj)]):
                if sent.lower() not in seen:
                    seen.add(sent.lower())
                    kept.append(sent)
        results.append(" ".join(kept))
    return results

outs = summarize_all([get_src(e) for e in test])
outs = [o if o else tokenizer.decode(get_src(e)[:300]) for o, e in zip(outs, test)]   # never submit empty
json.dump([{"id": e["id"], "summary": s} for e, s in zip(test, outs)], open("prediction.json", "w"))
with zipfile.ZipFile("encdec-full.zip", "w") as z:
    z.write("prediction.json", "prediction.json")
print("wrote", len(outs), "summaries")

Generating:   0%|          | 0/100 [00:00<?, ?item/s]

wrote 100 summaries
